---

# Chapter 0 — Research Foundation

# Insurance Claims Fraud Detection : A Data Driven Approach Using SQL and Machine Learning
## An Emperical Study of Fraud Patterns in Insurance Claims Using Database Querying and Predictive Modelling. 

---

| | |
|---|---|
| **Author** | Whitney Kemuma |
| **Degree** | BSc Actuarial Science |
| **Applying For** | Research Master's in [Data Science / Risk Analysis / Statistics / Machine Learning] |
| **Dataset** | Claims data from mastmustu/Kaggle - insurance_data.csv |
| **Tools** | PostgreSQL (SQL), Pandas, Python, Statsmodels, Scikit-learn, Statsmodels |
| **Year** | 2026 |
---

## 0.1 Research Problem

Insurance fraud is a systemic challenge costing the global insurance industry an estimated
**USD 80 billion annually** (FBI, 2023). Fraudulent claims inflate premiums, erode insurer
profitability, and penalise honest policyholders. Despite its scale, fraud detection in
insurance remains methodologically underdeveloped, particularly for multiline insurers
managing diverse product portfolios spanning Health, Motor, Life, Property, Mobile, and
Travel lines.

This research investigates whether **demographic, behavioural, and incident level features**
of insurance claims can reliably predict claim denial (used here as a fraud proxy), and whether
statistically significant differences exist in claim characteristics across insurance types
and risk segments.

---

## 0.2 Research Questions

> **Primary RQ:** What policyholder and claim characteristics are most strongly associated
> with claim denial in a multi-line insurance portfolio?

> **RQ2:** Is there a statistically significant difference in claim amounts across
> insurance types?

> **RQ3:** Does risk segmentation (H/M/L) significantly predict claim denial rate?

> **RQ4:** Do high-risk policyholders exhibit detectably different claim-filing behaviour
> (lag, severity, authority contacted) compared to low-risk policyholders?

---

## 0.3 Research Hypotheses

| # | Null Hypothesis (H₀) | Alternative (H₁) | Test |
|---|---|---|---|
| H1 | Mean claim amount is equal across all insurance types | At least one type differs significantly | One-way ANOVA |
| H2 | Claim denial rate is independent of risk segmentation | Denial rate differs by risk group | Chi-square test |
| H3 | Mean claim amount is equal for denied vs approved claims | Denied claims have different mean amounts | Independent t-test |
| H4 | Report lag (days) is equal across incident severity levels | Lag differs by severity | One-way ANOVA |

---

## 0.4 Phase 1 Summary — What Was Done in SQL (PostgreSQL)

This section documents all decisions made during the SQL phase, providing the methodological
continuity required in research.

### 0.4.1 Database Architecture

A **Third Normal Form (3NF)** relational schema was designed and implemented in PostgreSQL 16
comprising three tables:

- **`policyholders`** — 17 demographic and financial attributes per customer (PII excluded)
- **`policies`** — 6 policy-level attributes including insurance type and premium
- **`claims`** — 15 claim transaction attributes including dates, severity, and claim status

A **staging table** (`staging_claims`) was used as an import buffer — all 38 columns were
loaded as TEXT to prevent type casting errors during import. Data was then transformed and
loaded into the normalised tables via an ETL pipeline.

### 0.4.2 Data Quality Findings (SQL Audit)

| Column | Null Count | Decision |
|---|---|---|
| `ADDRESS_LINE2` | 8,505 (85%) | Retained as NULL — structural missingness (optional field) |
| `VENDOR_ID` | 3,245 (32%) | Retained as NULL — not all claims involve a vendor |
| `AUTHORITY_CONTACTED` | 1,945 (19%) | Retained as NULL — no authority contacted for those claims |
| `CUSTOMER_EDUCATION_LEVEL` | 529 (5.3%) | Retained as NULL — handled in Python as 'Unknown' |
| `CITY` | 54 (0.5%) | Retained as NULL — negligible impact |
| `INCIDENT_CITY` | 46 (0.5%) | Retained as NULL — negligible impact |

**No `?` placeholder values were found** — unlike the classic mastmustu dataset, this file
used proper NULL values, eliminating the need for string replacement cleaning.

**PII Exclusion:** `SSN`, `ROUTING_NUMBER`, and `ACCT_NUMBER` were loaded into staging
but intentionally excluded from the normalised schema, documented as a data governance decision.

### 0.4.3 SQL Analyses Conducted

Ten SQL scripts were written, committed to GitHub, and exported as CSV:

| Script | Analysis | Key Finding |
|---|---|---|
| `05_frequency_severity.sql` | Claim frequency & severity by insurance type | Denial rate varies by type |
| `06_loss_ratio.sql` | Loss ratio (claims/premiums × 100) by type | Loss ratios exceed 10,000% — actuarially significant |
| `07_high_risk.sql` | High-risk policyholder segmentation | H-segment has 1,455 customers |
| `08_fraud_detection.sql` | Window function fraud flag (claim > 2× type avg + no police report) | Rule-based flag validated against CLAIM_STATUS |
| `09_settlement_lag.sql` | Report and processing lag (LOSS_DT → REPORT_DT → TXN_DATE_TIME) | Avg 3-day report lag, 7-day processing lag |
| `10_cte_denied_risk_profile.sql` | CTE: denial rate by risk segment, social class, employment | H-segment shows higher denial concentration |

### 0.4.4 The Fraud Label

This dataset does not contain an explicit `fraud_reported` column. The variable **`CLAIM_STATUS`**
is used as the analytical target:

- **`A` (Approved):** 9,497 claims (94.97%)
- **`D` (Denied):** 503 claims (5.03%)

In insurance practice, claim denial is the outcome of underwriter assessment of claim
legitimacy, risk profile, and policy terms. Denied claims therefore serve as a
**validated proxy for high-risk or suspicious claim behaviour**, making `CLAIM_STATUS = 'D'`
a defensible fraud indicator for research purposes.

---

## 0.5 Phase 2 Overview — Python Workflow

| Chapter | Content |
|---|---|
| **Ch 0** | Research foundation, questions, hypotheses, SQL summary (this chapter) |
| **Ch 1** | Data importation, type casting, feature engineering, quality validation |
| **Ch 2** | Exploratory Data Analysis — distributions, correlations, denial patterns |
| **Ch 3** | Hypothesis testing — ANOVA, chi-square, t-tests |
| **Ch 4** | Visualisations — publication-quality charts exported to `reports/` |
| **Ch 5** | Fraud classification model — logistic regression + evaluation metrics |
| **Ch 6** | Conclusions, limitations, and recommendations |

---
